In [ ]:
# prompt: Impact of Social Media Trends on Business Growth for ds research project

# Install necessary libraries if not already installed
!pip install pandas matplotlib seaborn scikit-learn tweepy textblob

# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import tweepy
from textblob import TextBlob

# 1. Data Collection
#   - Collect data from social media platforms (Twitter, Facebook, Instagram, etc.)
#   - Use APIs or web scraping techniques
#   - Example using Tweepy for Twitter data:

# Authenticate with Twitter API
import os
consumer_key = os.environ.get("TWITTER_CONSUMER_KEY")
consumer_secret = os.environ.get("TWITTER_CONSUMER_SECRET")
access_token = os.environ.get("TWITTER_ACCESS_TOKEN")
access_token_secret = os.environ.get("TWITTER_ACCESS_TOKEN_SECRET")

auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_token, access_token_secret)
api = tweepy.API(auth)

# Search for tweets related to a specific trend
keyword = "AI"
tweets = api.search_tweets(q=keyword, count=100)

# 2. Data Preprocessing
#   - Clean the data (remove irrelevant information, handle missing values)
#   - Extract relevant features (e.g., number of likes, shares, comments, sentiment)
#   - Example:

data = []
for tweet in tweets:
  text = tweet.text
  sentiment = TextBlob(text).sentiment.polarity
  likes = tweet.favorite_count
  retweets = tweet.retweet_count
  data.append([text, sentiment, likes, retweets])

df = pd.DataFrame(data, columns=["Text", "Sentiment", "Likes", "Retweets"])

# 3. Exploratory Data Analysis
#   - Visualize the data to understand patterns and relationships
#   - Example:

sns.scatterplot(x="Sentiment", y="Likes", data=df)
plt.show()

# 4. Feature Engineering
#   - Create new features from existing ones to improve model performance
#   - Example:

df["Engagement"] = df["Likes"] + df["Retweets"]

# 5. Model Building
#   - Choose a suitable machine learning model (e.g., regression, classification)
#   - Train the model on the preprocessed data
#   - Example using Linear Regression:

X = df[["Sentiment", "Engagement"]]
y = df["Likes"]  # Assuming likes as a measure of business growth

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

# 6. Model Evaluation
#   - Evaluate the model's performance using appropriate metrics
#   - Example:

y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print("Root Mean Squared Error:", rmse)

# 7. Interpretation and Insights
#   - Analyze the model's results to gain insights into the impact of social media trends on business growth
#   - Communicate findings and recommendations to stakeholders


In [ ]:
#. Social Media Data Collection (Twitter Example using Tweepy):
import tweepy
import pandas as pd

# Twitter API credentials
import os
consumer_key = os.environ.get("TWITTER_CONSUMER_KEY")
consumer_secret = os.environ.get("TWITTER_CONSUMER_SECRET")
access_token = os.environ.get("TWITTER_ACCESS_TOKEN")
access_token_secret = os.environ.get("TWITTER_ACCESS_TOKEN_SECRET")

# Authenticate with Twitter API
auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_token, access_token_secret)
api = tweepy.API(auth)

# Function to fetch tweets
def fetch_tweets(query, max_tweets=1000):
    tweets = tweepy.Cursor(api.search_tweets, q=query, lang="en", tweet_mode="extended").items(max_tweets)
    tweets_data = [[tweet.created_at, tweet.user.screen_name, tweet.full_text] for tweet in tweets]
    return pd.DataFrame(tweets_data, columns=['Date', 'User', 'Tweet'])

# Fetch tweets related to ingredient transparency
tweets_df = fetch_tweets(query="P&G transparency", max_tweets=1000)
tweets_df.to_csv('tweets_data.csv', index=False)


In [ ]:
#Sales and Market Data (Example using CSV files)
import pandas as pd

# Load sales data
sales_df = pd.read_csv('pg_sales_data.csv')

# Load market share data
market_df = pd.read_csv('pg_market_share.csv')

# Merge sales and market data if needed
business_data = pd.merge(sales_df, market_df, on='Date')


In [ ]:
#2. Data Preprocessing a. Preprocessing Tweets (Text Cleaning)
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords
nltk.download('stopwords')

# Function to clean tweets
def clean_tweet(tweet):
    tweet = re.sub(r'http\S+', '', tweet)  # Remove URLs
    tweet = re.sub(r'@\w+', '', tweet)     # Remove mentions
    tweet = re.sub(r'#\w+', '', tweet)     # Remove hashtags
    tweet = re.sub(r'\W', ' ', tweet)      # Remove special characters
    tweet = tweet.lower()                  # Convert to lowercase
    tweet = ' '.join([word for word in tweet.split() if word not in stopwords.words('english')])
    return tweet

tweets_df['Cleaned_Tweet'] = tweets_df['Tweet'].apply(clean_tweet)


In [ ]:
#3. Sentiment Analysis a. Sentiment Analysis Using VADER:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Initialize VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Function to get sentiment scores
def get_sentiment(tweet):
    return sia.polarity_scores(tweet)['compound']

tweets_df['Sentiment'] = tweets_df['Cleaned_Tweet'].apply(get_sentiment)

# Aggregate sentiment over time
sentiment_trend = tweets_df.groupby('Date')['Sentiment'].mean().reset_index()


In [ ]:
#4. Correlation Analysis a. Correlating Social Media Sentiment with Sales Data
# Merge sentiment trend with sales data
analysis_df = pd.merge(sentiment_trend, business_data, on='Date')

# Correlation analysis
correlation = analysis_df[['Sentiment', 'Sales']].corr()
print(correlation)


In [ ]:
#5. Predictive Modeling a. Time Series Forecasting Using ARIMA
from statsmodels.tsa.arima.model import ARIMA

# Prepare data for modeling
sales_series = analysis_df.set_index('Date')['Sales']
sentiment_series = analysis_df.set_index('Date')['Sentiment']

# Build ARIMA model using sales data
model = ARIMA(sales_series, order=(5, 1, 0))
model_fit = model.fit()

# Forecast future sales
forecast = model_fit.forecast(steps=30)
print(forecast)

# b. Predictive Modeling Using Linear Regression:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Prepare data
X = analysis_df[['Sentiment']].values
y = analysis_df['Sales'].values

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict sales
y_pred = model.predict(X_test)


In [ ]:
# Visualization a. Visualizing Sentiment Trends and Sales:
import matplotlib.pyplot as plt

# Plot sentiment trend
plt.figure(figsize=(12, 6))
plt.plot(sentiment_trend['Date'], sentiment_trend['Sentiment'], label='Sentiment Trend')
plt.xlabel('Date')
plt.ylabel('Sentiment')
plt.title('Social Media Sentiment Trend')
plt.legend()
plt.show()

# Plot sales data
plt.figure(figsize=(12, 6))
plt.plot(business_data['Date'], business_data['Sales'], label='Sales Data', color='orange')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Sales Trend')
plt.legend()
plt.show()


In [ ]:
#7. Interactive Dashboard
import streamlit as st

# Streamlit app
st.title("Social Media Trends Impact on Business Growth")

# Plot sentiment trend
st.line_chart(sentiment_trend.set_index('Date')['Sentiment'])

# Plot sales trend
st.line_chart(business_data.set_index('Date')['Sales'])

# Display correlation
st.write(f"Correlation between sentiment and sales: {correlation.iloc[0, 1]:.2f}")


# 8. Deployment and Monitoring. Can deploy the predictive models and dashboard on a cloud platform (e.g., AWS, Heroku) and set up monitoring to track real-time social media sentiment and business performance.